<a href="https://colab.research.google.com/github/Yennanng/PTDLNC/blob/develop/ADA_Diagnostics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import gdown
import time

# Tải dữ liệu
file_id = '17Emm31vo3KDA3q4Djxp-0CKiClCAliPG'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'flights.csv'

# Chỉ tải nếu file chưa tồn tại để tiết kiệm thời gian
import os
if not os.path.exists(output):
    gdown.download(url, output, quiet=False)

# Khởi tạo biến df_flights quan trọng
df_flights = pd.read_csv(output, low_memory=False)
print(f"Đã nạp dữ liệu thành công. Kích thước: {df_flights.shape}")

In [ ]:
# 1. Import toàn bộ thư viện cần thiết
import pandas as pd
import numpy as np
import time
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from pandas.tseries.holiday import USFederalHolidayCalendar

# 2. Khởi tạo Scaler
scaler = RobustScaler()

# 3. Định nghĩa các hàm lấy dữ liệu (Data Generators)
def get_orig():
    return scaler.fit_transform(X_train_f), y_train_t, None

def get_weight():
    w = y_train_t.map({0: 1, 1: (len(y_train_t) - sum(y_train_t)) / sum(y_train_t)})
    return scaler.fit_transform(X_train_f), y_train_t, w

def get_down():
    d = pd.concat([X_train_f, y_train_t], axis=1)
    d_min = d[d['IS_DELAYED'] == 1]
    d_maj = d[d['IS_DELAYED'] == 0].sample(n=len(d_min), random_state=42)
    d_bal = pd.concat([d_maj, d_min])
    return scaler.fit_transform(d_bal.drop(columns=['IS_DELAYED'])), d_bal['IS_DELAYED'], None

# 4. Định nghĩa danh sách mô hình
models = {
    "Logistic": LogisticRegression(max_iter=1000),
    "DecisionTree": DecisionTreeClassifier(max_depth=10),
    "RandomForest": RandomForestClassifier(n_estimators=10, max_depth=10, n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=100, random_state=42, verbosity=-1),
    "XGBoost": XGBClassifier(n_estimators=100, eval_metric='logloss')
}

print("✅ Đã import thư viện và khởi tạo Models/Scaler thành công!")

In [ ]:
# 1. Chuẩn hóa tên cột (Viết hoa và xóa khoảng trắng thừa)
df_flights.columns = df_flights.columns.str.upper().str.strip()

# 2. Tiền xử lý rút gọn
# Lọc bỏ chuyến bay không thực hiện
df_active = df_flights[df_flights.get(['CANCELLED', 'DIVERTED'], 0).sum(axis=1) == 0].copy()

# Tạo cột DATE nhanh
df_active['DATE'] = pd.to_datetime({'year': 2015, 'month': df_active['MONTH'], 'day': df_active['DAY']})

# Feature Engineering
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start='2015-01-01', end='2015-12-31')
df_active['IS_WEEKEND'] = (df_active['DATE'].dt.weekday >= 5).astype(int)
df_active['IS_HOLIDAY'] = df_active['DATE'].isin(holidays).astype(int)

# Tạo nhãn và loại bỏ cột dư thừa/gây rò rỉ dữ liệu
df_real = df_active.copy()
df_real['IS_DELAYED'] = (df_active['ARRIVAL_DELAY'] > 15).astype(int)

# Danh sách loại bỏ (chỉ bỏ những cột tồn tại)
cols_to_drop = [
    'YEAR', 'FLIGHT_NUMBER', 'TAIL_NUMBER', 'CANCELLATION_REASON', 'DATE',
    'DEPARTURE_DELAY', 'DEPARTURE_TIME', 'WHEELS_OFF', 'WHEELS_ON',
    'ARRIVAL_TIME', 'AIR_TIME', 'TAXI_IN', 'TAXI_OUT', 'ARRIVAL_DELAY',
    'ACTUAL_ELAPSED_TIME', 'ELAPSED_TIME', 'CANCELLED', 'DIVERTED'
]
# Thêm các cột delay chi tiết nếu có
cols_to_drop += [c for c in df_real.columns if 'DELAY' in c and c != 'IS_DELAYED']

df_real = df_real.drop(columns=[c for c in set(cols_to_drop) if c in df_real.columns])

print(f"✅ Xử lý xong! Kích thước: {df_real.shape}")
print(f"Các biến dự báo: {df_real.columns.tolist()}")

## **DESCRIPTIVE ANALYSIS**

### **1. Tổng quan dataset & biến mục tiêu**

In [ ]:
print("📊 TỔNG QUAN DỮ LIỆU")
print("--------------------------------------------------")
print(f"Số dòng: {df_real.shape[0]:,}")
print(f"Số cột: {df_real.shape[1]}")
print("\nTỷ lệ biến mục tiêu (IS_DELAYED):")
display(df_real['IS_DELAYED'].value_counts(normalize=True).rename("Ratio"))


### **2. Thống kê mô tả các biến số**

In [ ]:
print("📈 THỐNG KÊ MÔ TẢ CÁC BIẾN SỐ")
display(df_real.describe().T)

### **3. Phân phối chuyến bay theo thời gian**

Theo tháng

In [ ]:
flights_by_month = df_active.groupby('MONTH').size()

flights_by_month.plot(
    kind='bar',
    title='Số lượng chuyến bay theo tháng',
    figsize=(10,5)
)

Tỷ lệ trễ theo tháng

In [ ]:
delay_rate_month = df_real.groupby('MONTH')['IS_DELAYED'].mean()

delay_rate_month.plot(
    kind='line',
    marker='o',
    title='Tỷ lệ trễ chuyến theo tháng',
    figsize=(10,5)
)

### **4. Ngày thường vs Cuối tuần**

In [ ]:
weekend_summary = df_real.groupby('IS_WEEKEND')['IS_DELAYED'].agg(
    Total_Flights='count',
    Delay_Rate='mean'
)

display(weekend_summary)

### **5. Ngày lễ vs Ngày thường**

In [ ]:
holiday_summary = df_real.groupby('IS_HOLIDAY')['IS_DELAYED'].agg(
    Total_Flights='count',
    Delay_Rate='mean'
)

display(holiday_summary)


### **6. Phân tích theo hãng hàng không**

In [ ]:
airline_summary = (
    df_real
    .groupby('AIRLINE')['IS_DELAYED']
    .agg(
        Total_Flights='count',
        Delay_Rate='mean'
    )
    .sort_values('Delay_Rate', ascending=False)
)

display(airline_summary.head(10))


### **7. Cuối tuần ảnh hưởng đến hãng**

In [ ]:
airline_weekend_effect = (
    df_real
    .groupby(['AIRLINE', 'IS_WEEKEND'])['IS_DELAYED']
    .mean()
    .unstack()
)

airline_weekend_effect['Weekend_Effect'] = (
    airline_weekend_effect[1] - airline_weekend_effect[0]
)

display(airline_weekend_effect.sort_values('Weekend_Effect', ascending=False))


### **8. Phân tích theo sân bay (Top bận nhất)**

Sân bay xuất phát

In [ ]:
origin_summary = (
    df_real
    .groupby('ORIGIN_AIRPORT')['IS_DELAYED']
    .agg(
        Total_Flights='count',
        Delay_Rate='mean'
    )
    .query("Total_Flights > 1000")
    .sort_values('Delay_Rate', ascending=False)
)

display(origin_summary.head(10))


### **9. Phân tích rủi ro trễ không phụ thuộc lưu lượng**

In [ ]:
airport_risk = (
    df_real
    .groupby('ORIGIN_AIRPORT')['IS_DELAYED']
    .agg(
        Total_Flights='count',
        Delay_Rate='mean'
    )
    .query("Total_Flights > 1000")
)

display(airport_risk.sort_values('Delay_Rate', ascending=False))

### **10. Correlation giữa các biến số**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

num_cols = [
    'MONTH', 'DAY', 'IS_WEEKEND', 'IS_HOLIDAY', 'IS_DELAYED'
]

plt.figure(figsize=(8,6))
sns.heatmap(
    df_real[num_cols].corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)
plt.title("Correlation Matrix (Descriptive Analytics)")
plt.show()

## **DIAGNOSTIC ANALYSIS**

**3.1. Diagnostic theo thời gian**

In [ ]:
monthly_summary = (
    df_real
    .groupby('MONTH')
    .agg(
        Total_Flights=('IS_DELAYED', 'count'),
        Delay_Rate=('IS_DELAYED', 'mean')
    )
    .reset_index()
)

monthly_summary


In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(10,5))

ax1.bar(
    monthly_summary['MONTH'],
    monthly_summary['Total_Flights'],
    alpha=0.6
)
ax1.set_xlabel('Month')
ax1.set_ylabel('Total Flights')

ax2 = ax1.twinx()
ax2.plot(
    monthly_summary['MONTH'],
    monthly_summary['Delay_Rate'],
    marker='o'
)
ax2.set_ylabel('Delay Rate')

plt.title('Seasonality Diagnostic: Total Flights vs Delay Rate')
plt.show()
